<a href="https://colab.research.google.com/github/prithwis/parashar21/blob/main/K21_TwoPass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/parashar21/main/images/khona21-v0.png) <br>


---



[Prithwis Mukerjee](http://www.yantrajaal.com) ||
Website - [Parashar21](https://parashar21.blogspot.com)<br>

# Khona21 — AI-Assisted Jyotiṣa Interpretation

**Khona21** is an experimental AI-based Jyotiṣa interpretation platform, evolved from the original **Parashar21** project, which explored the application of artificial intelligence to traditional Indian astrology.

Named after the legendary Bengali astrologer Khona, the platform uses Large Language Models (LLMs) to interpret precomputed horoscopes against a curated corpus of Jyotiṣa literature, currently comprising works by **James Braha** and **Harihar Majumdar**.

Rather than relying on predefined interpretive rules, Khona21 uses source-grounded AI reasoning to identify planetary configurations, yogas, exceptions and conflicting interpretations, and synthesizes them into responses to specific questions.

This notebook, **k21_KonsultKhona**, demonstrates the platform's consultation workflow.

Khona21 is an experiment in applying modern AI to traditional knowledge systems, not a scientific validation of astrological predictions.

In [1]:
!pip -q install pypandoc
!pip -q install python-docx
from IPython.display import display, Markdown

!wget -O k21core.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/k21utils/k21core.py
!wget -O k21fileops.py -q https://raw.githubusercontent.com/prithwis/parashar21/main/k21utils/k21fileops.py
from k21core import *
from k21fileops import *

In [ ]:
# ------------------------------------------------
# required for multiple imports of the same module
# ------------------------------------------------
#import k21core
#import importlib

#importlib.reload(k21core)
from k21core import *

In [2]:
# ------------------------------------------------
# Set Parameters
# ------------------------------------------------
provider = "openai"

#model = "gpt-5.6-luna"
model = "gpt-6-luna"
#model = "gpt-6-astra"

id = "TG"

###Access Control
This notebook accesses Google Drive to access files and uses the OpenAI API. Hence it can be used ONLY from the userid of the creator. Others will have to create their own access paths

In [3]:
# ------------------------------------------------
# Authenticate wit LLM provider using API keys stored as Colab Secret
# ------------------------------------------------
client = authenticateLLM(provider=provider)
# ------------------------------------------------
# Locate necessary file and set up paths to the same
# ------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')
#ls /content/drive/MyDrive/Prithwis/Parashar21/K21

K21_HOME = "/content/drive/MyDrive/Prithwis/Parashar21/K21"

# ------------------------------------------------
# The Corpus Files, Braha and Harihar, are available
# in GitHub Repo and can be downloaded with wget
# ------------------------------------------------
BRAHA   = f"{K21_HOME}/Corpus/BR_Curated_22.txt"
HARIHAR = f"{K21_HOME}/Corpus/HM_Clean_07.txt"
# ------------------------------------------------
# The Chart, Navamsha and Dasha data are generated by Parashar21
# Needs to be brought into this VM
# ------------------------------------------------
CHART   = f"{K21_HOME}/Natives/{id}-Chart.txt"
NAVAMSA = f"{K21_HOME}/Natives/{id}-Navamsa_Chart.txt"
DASHA   = f"{K21_HOME}/Natives/{id}-Dasha.txt"

OPENAI | User: Calcutta | Email: prithwis@yantrajaal.com | ID: user-sNAljKNmBf2igtAV2kUuqhoF
Mounted at /content/drive


In [4]:
# ------------------------------------------------
# Define the ROLE
# ------------------------------------------------
RoleFile = f"{K21_HOME}/Roles/khona261027.txt"
khona21 = readText(RoleFile)

/content/drive/MyDrive/Prithwis/Parashar21/K21/Roles/khona261027.txt | 5,554 characters


In [5]:
# ------------------------------------------------
# Set the Context
# ------------------------------------------------
#BaseCONTEXT = buildContext(BRAHA, HARIHAR, CHART, NAVAMSA, DASHA)
BaseCONTEXT = getContext(
    BRAHA=BRAHA,
    HARIHAR=HARIHAR,
    CHART=CHART,
    NAVAMSA=NAVAMSA,
    DASHA=DASHA
)

In [7]:
#BaseCONTEXT

In [8]:
# ------------------------------------------------
# FIRST PASS
#
# ------------------------------------------------

FirstPass_Instruction = """
FIRST PASS — FOUNDATIONAL ASSESSMENT

Prepare a concise, high-level assessment of this horoscope.

Present an overall picture of the native's principal
strengths, weaknesses and significant planetary configurations.

Identify the most important favourable and unfavourable yogas,
sign exchanges, cancellations and mitigating factors,
to the extent that they are evident from the supplied data
and reference texts.

Begin with a simple-language summary, followed by a brief
astrological explanation with relevant source references.

This is an initial orientation, not an exhaustive audit.
Focus on the dominant features of the horoscope rather than
attempting to catalogue every planetary condition or yoga.

Distinguish chart facts from interpretations, and acknowledge
important contradictions or uncertainties.

Do not invent configurations or use rules unsupported by
the supplied reference texts.

Do not make predictions about specific future events.
"""

In [9]:
# ------------------------------------------------
# Assemble the first pass query
# Call the LLM
# ------------------------------------------------

QUERY = buildQuery(BaseCONTEXT, FirstPass_Instruction)

FirstPass = callLLM(
    client=client,
    role=khona21,
    query=QUERY,
    model=model,
    provider=provider,
    effort="medium"
)


OPENAI | gpt-6-luna | Effort: medium | In: 222,424 | Cached: 0 | Out: 7,451 | Reasoning: 5,564 | Time: 84.6s


In [11]:
#display(Markdown(FirstPass))
# ------------------------------------------------
# Save answer as markdown for machine use
# ------------------------------------------------
#MD_FILE = saveAnswer(id, model, FirstPass)

In [ ]:
def augmentContext(baseContext, firstPass):
    """
    Augment the original context with the FirstPass horoscope assessment.
    Returns a new context without modifying the original.
    """
    return (
        baseContext
        + "\n\n<FOUNDATIONAL_ASSESSMENT>\n"
        + firstPass
        + "\n</FOUNDATIONAL_ASSESSMENT>\n"
    )

In [13]:
CONTEXT = augContext(BaseCONTEXT, FIRSTPASS=FirstPass)

In [14]:
# ------------------------------------------------
# Ask your question / define the task
# This and subsequent cells can be run multiple times
# ------------------------------------------------
CurrentQuestion = """
Will the native publish a book? If so, would it find takers ? If so, would it be a novel or a text book?
"""

In [16]:
# ------------------------------------------------
# Assemble the query
# Call the LLM
# ------------------------------------------------

QUERY = buildQuery(CONTEXT, CurrentQuestion)

answer = callLLM(
    client=client,
    role=khona21,
    query=QUERY,
    model=model,
    provider=provider,
    effort="medium"
)

OPENAI | gpt-6-luna | Effort: medium | In: 224,175 | Cached: 1,098 | Out: 8,860 | Reasoning: 7,768 | Time: 95.8s


In [17]:
#len(BaseCONTEXT), len(FirstPass), len(context)

Saved: /content/K21_TG_gpt-6-luna_261027_140910.md


In [18]:
# ------------------------------------------------
# Save answer as markdown for machine use
# ------------------------------------------------
MD_FILE = saveAnswer(id, model, answer)
# ------------------------------------------------
# Convert markdown to docx for human readability
# Second Parameter is identifier of native, can be replaced by id
# ------------------------------------------------
DOCX_FILE = markdownToDocx(MD_FILE, "TopGun", CurrentQuestion, provider,model)

Saved: /content/K21_TG_gpt-6-luna_261027_193626.md
Saved: /content/K21_TG_gpt-6-luna_261027_193626.docx


In [19]:
Gochar = """
GOCHAR DATA FOR JYOTISHA REASONING
=====================================

The house lordships and functional benefic/malefic labels in the Gochar chart are
calculated relative to Chandra Lagna and must not replace the natal Rashi chart's lordships and functional classifications.

Analysis : Gochar
Chart Type : Rashi
Reference Date : 27-Oct-2026
Gender : x

HOUSE LORDSHIP
--------------
House 1 lord is Jupiter
House 2 lord is Saturn
House 3 lord is Saturn
House 4 lord is Jupiter
House 5 lord is Mars
House 6 lord is Venus
House 7 lord is Mercury
House 8 lord is Moon
House 9 lord is Sun
House 10 lord is Mercury
House 11 lord is Venus
House 12 lord is Mars

PLANETARY INFORMATION
---------------------

Lagna:
- Occupies Bhava 1
- Located in Rashi Sagittarius
- Located in Nakshatra 19 (Mula)
- Absolute celestial longitude 250.52 degrees

Sun:
- Occupies Bhava 11
- Located in Rashi Libra
- Located in Nakshatra 15 (Swati)
- Lord of Bhavas [9]
- Debilitated
- In enemy house
- Functional benefic
- Absolute celestial longitude 189.943 degrees

Moon:
- Occupies Bhava 5
- Located in Rashi Aries
- Located in Nakshatra 3 (Krittika)
- Lord of Bhavas [8]
- Absolute celestial longitude 29.138 degrees

Mars:
- Occupies Bhava 8
- Located in Rashi Cancer
- Located in Nakshatra 9 (Ashlesha)
- Lord of Bhavas [5, 12]
- Debilitated
- In friendly house
- Functional benefic
- Absolute celestial longitude 112.139 degrees

Mercury:
- Occupies Bhava 11
- Located in Rashi Libra
- Located in Nakshatra 16 (Vishakha)
- Lord of Bhavas [7, 10]
- In friendly house
- Functional malefic
- Retrograde
- Absolute celestial longitude 205.983 degrees

Jupiter:
- Occupies Bhava 8
- Located in Rashi Cancer
- Located in Nakshatra 9 (Ashlesha)
- Lord of Bhavas [1, 4]
- Exalted
- In friendly house
- Absolute celestial longitude 119.533 degrees

Venus:
- Occupies Bhava 11
- Located in Rashi Libra
- Located in Nakshatra 14 (Chitra)
- Lord of Bhavas [6, 11]
- In own house
- In friendly house
- Functional malefic
- Retrograde
- Absolute celestial longitude 184.437 degrees

Saturn:
- Occupies Bhava 4
- Located in Rashi Pisces
- Located in Nakshatra 26 (UttaraBhadrapada)
- Lord of Bhavas [2, 3]
- Retrograde
- Absolute celestial longitude 345.342 degrees

Rahu:
- Occupies Bhava 3
- Located in Rashi Aquarius
- Located in Nakshatra 23 (Dhanishtha)
- Functional malefic
- Absolute celestial longitude 302.075 degrees

Ketu:
- Occupies Bhava 9
- Located in Rashi Leo
- Located in Nakshatra 10 (Magha)
- Functional malefic
- Absolute celestial longitude 122.075 degrees

PLANETARY ASPECTS
-----------------
Sun aspects ['Moon']
Moon aspects ['Venus', 'Sun', 'Mercury']
Mars aspects ['Rahu', 'Venus', 'Sun', 'Mercury']
Mercury aspects ['Moon']
Jupiter aspects ['Saturn']
Venus aspects ['Moon']
Saturn aspects ['Lagna']

PLANETS ASPECTED BY
--------------------
Moon is aspected by ['Venus', 'Sun', 'Mercury']
Sun is aspected by ['Moon', 'Mars']
Mercury is aspected by ['Moon', 'Mars']
Venus is aspected by ['Moon', 'Mars']
Rahu is aspected by ['Mars']
Saturn is aspected by ['Jupiter']
Lagna is aspected by ['Saturn']

PLANETARY CONJUNCTIONS
-----------------------
Sun conjuncts ['Venus', 'Mercury']
Mercury conjuncts ['Venus', 'Sun']
Venus conjuncts ['Sun', 'Mercury']
Mars conjuncts ['Jupiter']
Jupiter conjuncts ['Mars']

HOUSE LORD RELATIONSHIPS
-------------------------
Lord of Bhava 1 conjuncts ['Mars']
Lord of Bhava 4 conjuncts ['Mars']
Lord of Bhava 5 conjuncts ['Jupiter']
Lord of Bhava 6 conjuncts ['Sun', 'Mercury']
Lord of Bhava 7 conjuncts ['Venus', 'Sun']
Lord of Bhava 9 conjuncts ['Venus', 'Mercury']
Lord of Bhava 10 conjuncts ['Venus', 'Sun']
Lord of Bhava 11 conjuncts ['Sun', 'Mercury']
Lord of Bhava 12 conjuncts ['Jupiter']

LORD LORD RELATIONSHIPS
-----------------------
Lord of Bhava 1 connected with Lords [12, 5]
Lord of Bhava 4 connected with Lords [12, 5]
Lord of Bhava 5 connected with Lords [4, 1]
Lord of Bhava 6 connected with Lords [7, 9, 10]
Lord of Bhava 7 connected with Lords [9, 11, 6]
Lord of Bhava 9 connected with Lords [7, 10, 11, 6]
Lord of Bhava 10 connected with Lords [9, 11, 6]
Lord of Bhava 11 connected with Lords [7, 9, 10]
Lord of Bhava 12 connected with Lords [4, 1]

BHAVA ASPECTS
--------------
Bhava 1 receives aspect from ['Saturn']
Bhava 2 receives aspect from ['Jupiter', 'Mars']
Bhava 3 receives aspect from ['Ketu', 'Mars']
Bhava 4 receives aspect from ['Jupiter']
Bhava 5 receives aspect from ['Venus', 'Sun', 'Mercury']
Bhava 6 receives aspect from ['Saturn']
Bhava 7 receives aspect from ['Lagna']
Bhava 9 receives aspect from ['Rahu']
Bhava 10 receives aspect from ['Saturn']
Bhava 11 receives aspect from ['Moon', 'Mars']
Bhava 12 receives aspect from ['Jupiter']

BHAVA ASPECTED BY LORDS
------------------------
Bhava 1 receives aspect from Lords ['3', '2']
Bhava 2 receives aspect from Lords ['4', '5', '1', '12']
Bhava 3 receives aspect from Lords ['12', '5']
Bhava 4 receives aspect from Lords ['4', '1']
Bhava 5 receives aspect from Lords ['7', '6', '10', '9', '11']
Bhava 6 receives aspect from Lords ['3', '2']
Bhava 10 receives aspect from Lords ['3', '2']
Bhava 11 receives aspect from Lords ['12', '5', '8']
Bhava 12 receives aspect from Lords ['4', '1']

"""

In [20]:
CONTEXT = augContext(
    BaseCONTEXT,
    FIRSTPASS=FirstPass,
    GOCHAR=Gochar
)

In [21]:
# ------------------------------------------------
# Assemble the query
# Call the LLM
# ------------------------------------------------

QUERY = buildQuery(CONTEXT, CurrentQuestion)

answer = callLLM(
    client=client,
    role=khona21,
    query=QUERY,
    model=model,
    provider=provider,
    effort="medium"
)

OPENAI | gpt-6-luna | Effort: medium | In: 225,742 | Cached: 1,098 | Out: 10,357 | Reasoning: 8,701 | Time: 102.6s


In [22]:
# ------------------------------------------------
# Save answer as markdown for machine use
# ------------------------------------------------
MD_FILE = saveAnswer(id, model, answer)
# ------------------------------------------------
# Convert markdown to docx for human readability
# Second Parameter is identifier of native, can be replaced by id
# ------------------------------------------------
DOCX_FILE = markdownToDocx(MD_FILE, "TopGun", CurrentQuestion, provider,model)

Saved: /content/K21_TG_gpt-6-luna_261027_195251.md
Saved: /content/K21_TG_gpt-6-luna_261027_195251.docx


#ChronoTantra | ChronoYantra | ChronoMantra | ChronoSutra
 [More Information ](http://bit.ly/chronobooks) <br>

[![More Information](https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/CTYMS-Colab-Footer.png)](https://chronos.yantrajaal.com)